# Notebook 11 — imv_start sliding windows

### What this notebook does

We build the model input windows again, but in a more realistic way.

**Before:** windows were placed using the extubation time (`imv_end`). In real life we do not know when a patient will be extubated, so that design cannot run live.

**Now:** windows are placed using the ventilation start time (`imv_start`). This is known the moment the patient goes on the ventilator, so the model could actually be used at the bedside. This change also brings back patients the old design dropped (patients who were never extubated, mostly very sick ones).

**Settings we use:**
- Look-back (context): 48 hours
- Wait before the first window (warm-up): 48 hours
- Gap between windows (stride): 6 hours
- Most windows per patient: 8
- Keep non-survivor patients: yes

The output files match Notebook 07, so the tier notebooks can read them with only a path change.

### Setup and file paths

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_DIR    = Path('/content/drive/MyDrive/ventilator_weaning')
COHORT_DIR     = PROJECT_DIR / 'cohort'
PREPROCESS_DIR = PROJECT_DIR / 'preprocessing'
WINDOWS_DIR    = PROJECT_DIR / 'windows'
WINDOWS_DIR.mkdir(parents=True, exist_ok=True)

# Inputs we reuse (already cleaned, not rebuilt)
COHORT_PATH     = COHORT_DIR / 'cohort_zappala.parquet'
CLEAN_LONG_PATH = PREPROCESS_DIR / 'cohort_features_long_clean.parquet'

# New outputs (do not overwrite Notebook 07)
GRID_PATH     = WINDOWS_DIR / 'windows_tier12_grid_imvstart.npz'
TRIPLETS_PATH = WINDOWS_DIR / 'windows_tier3_triplets_imvstart.parquet'
METADATA_PATH = WINDOWS_DIR / 'windows_metadata_imvstart.json'

for p in [COHORT_PATH, CLEAN_LONG_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing {p}. Run notebooks 04 and 06 first.')

print('Inputs found.')

Mounted at /content/drive
Inputs found.


### Imports

In [2]:
import pandas as pd
import numpy as np
import json

### 1. Settings

In [3]:
CONTEXT_HOURS         = 48   # hours of history the model sees
WARMUP_HOURS          = 48   # first window this many hours after imv_start
STRIDE_HOURS          = 6    # gap between windows
MAX_WINDOWS_PER_VISIT = 8    # cap per patient
HORIZONS              = [1, 4, 8]

### 2. Load the cleaned data and add the anchor times

In [4]:
long_clean = pd.read_parquet(CLEAN_LONG_PATH)

cohort = pd.read_parquet(COHORT_PATH)
anchor = cohort[['visit_occurrence_id', 'imv_start', 'imv_end',
                 'imv_duration_hours', 'admission_end']].drop_duplicates()

df = long_clean.merge(anchor, on='visit_occurrence_id', how='inner')
df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'], utc=True)
df['imv_start'] = pd.to_datetime(df['imv_start'], utc=True)

# Hours counted from ventilation start
df['hours_since_imv_start'] = (df['measurement_datetime'] - df['imv_start']).dt.total_seconds() / 3600

print(f'Rows:     {len(df):,}')
print(f'Patients: {df["visit_occurrence_id"].nunique():,}')
print(f'Features: {df["feature_name"].nunique()}')

Rows:     5,925,778
Patients: 2,638
Features: 13


### 3. Check: patients we keep that the old design dropped

The old design removed patients whose ventilation ended right as they left the ICU (mostly deaths). Here we count how many such patients are in our data now. These are the patients the new design brings back.

In [5]:
anchor2 = anchor.copy()
anchor2['imv_end'] = pd.to_datetime(anchor2['imv_end'], utc=True)
anchor2['admission_end'] = pd.to_datetime(anchor2['admission_end'], utc=True)

# Hours between ventilation end and leaving the ICU
gap = (anchor2['admission_end'] - anchor2['imv_end']).dt.total_seconds() / 3600
would_be_dropped = gap <= 8

print(f'Total patients:                 {len(anchor2):,}')
print(f'Old design would have dropped:  {would_be_dropped.sum():,}')
print(f'We keep them now.')

Total patients:                 2,638
Old design would have dropped:  641
We keep them now.


### 4. Sanity check: do the values still look realistic?

This group now includes patients who died or were never extubated, so we expect some more extreme values. This check confirms those extremes are real clinical readings. Need to filter 0 values for heart and respiratory rate.

In [6]:
ranges = df.groupby('feature_name')['value_as_number'].describe(
    percentiles=[0.01, 0.5, 0.99])[['min', '1%', '50%', '99%', 'max']]

print(ranges.round(1).to_string())

                   min     1%    50%    99%     max
feature_name                                       
diastolic_bp      10.0   36.0   60.0   98.0   200.0
fio2              21.0   25.0   41.0   94.0   100.0
heart_rate         0.0   48.0   88.0  139.0   270.0
lactate            0.2    0.5    1.3   12.4    25.0
mean_bp           20.0   53.0   81.0  129.0   200.0
paco2              1.1    3.5    5.6   10.5    14.9
pao2               2.1    4.4   11.7   35.6    78.9
peep               0.0    3.0    8.0   19.0    25.0
ph                 6.5    7.1    7.4    7.5     7.7
respiratory_rate   0.0    9.0   22.0   44.0   100.0
spo2              50.0   87.0   97.0  100.0   100.0
systolic_bp       20.0   78.0  127.0  200.0   350.0
tidal_volume      50.0  129.0  439.0  967.0  1993.0


The measurements are generally within clinically realistic ranges, with a few extreme values (such as heart rate and respiratory rate of 0).

### 5. Pick the sliding windows for each patient

AI-assisted: per-patient sliding-window selection with an even-spacing cap, generated with Claude Opus 4.8 and verified by the author.


In [7]:
max_h = max(HORIZONS)
duration = anchor.set_index('visit_occurrence_id')['imv_duration_hours']


def pick_origins(dur_hours):
    # last window must leave room for the +8h target inside the episode
    last = int(np.floor(dur_hours)) - max_h
    if last < WARMUP_HOURS:
        return []

    origins = list(range(WARMUP_HOURS, last + 1, STRIDE_HOURS))

    # cap at 8 windows per patient so long-stay patients don't dominate the dataset
    if len(origins) > MAX_WINDOWS_PER_VISIT:
        pick = np.linspace(0, len(origins) - 1, MAX_WINDOWS_PER_VISIT).round().astype(int)
        origins = sorted({origins[i] for i in pick})

    return origins


origins_by_visit = {v: pick_origins(d) for v, d in duration.items()}
origins_by_visit = {v: o for v, o in origins_by_visit.items() if o}

print(f'Eligible patients: {len(origins_by_visit):,}')
print(f'Total windows:     {sum(len(o) for o in origins_by_visit.values()):,}')

Eligible patients: 1,912
Total windows:     13,017


### 6. Make a table of all windows

In [8]:
rows = []
for visit_id, origins in origins_by_visit.items():
    for t in origins:
        rows.append({'visit_occurrence_id': visit_id, 'origin_t': t,
                     'context_start': t - CONTEXT_HOURS, 'context_end': t})

windows = pd.DataFrame(rows)
windows['window_id'] = windows['visit_occurrence_id'].astype(str) + '_t' + windows['origin_t'].astype(str)

print(f'Windows: {len(windows):,}  from  {windows["visit_occurrence_id"].nunique():,} patients')
print(windows.head())

Windows: 13,017  from  1,912 patients
   visit_occurrence_id  origin_t  context_start  context_end window_id
0                   15        48              0           48    15_t48
1                   15        72             24           72    15_t72
2                   15        96             48           96    15_t96
3                   15       120             72          120   15_t120
4                   15       150            102          150   15_t150


The final dataset contains 13,017 forecasting windows from 1,912 patients, with each window using the previous 48 hours of data to predict the future

### 7. List the hours each window needs

AI-assisted: collect only the hours used by the context and targets, generated with Claude Opus 4.8 and verified by the author.


In [9]:
FEATURES = sorted(df['feature_name'].unique())

needed = []
for r in windows.itertuples(index=False):
    for h in range(r.context_start, r.context_end):   # 48 context hours
        needed.append((r.visit_occurrence_id, h))
    for h in [r.origin_t + hz for hz in HORIZONS]:     # 3 target hours
        needed.append((r.visit_occurrence_id, h))

needed = pd.DataFrame(needed, columns=['visit_occurrence_id', 'hour_bin']).drop_duplicates()
grid_cells = needed.merge(pd.DataFrame({'feature_name': FEATURES}), how='cross')

print(f'Grid cells to build: {len(grid_cells):,}')

Grid cells to build: 4,826,419


### 8. Mark which hours were actually measured

In [10]:
eligible = sorted(windows['visit_occurrence_id'].unique())

sub = df[df['visit_occurrence_id'].isin(eligible)].copy()
sub['hour_bin'] = np.floor(sub['hours_since_imv_start']).astype(int)

observed = sub.groupby(['visit_occurrence_id', 'feature_name', 'hour_bin'],
                       as_index=False)['value_as_number'].mean()

master = grid_cells.merge(observed, on=['visit_occurrence_id', 'feature_name', 'hour_bin'], how='left')
master['is_observed'] = master['value_as_number'].notna().astype(int)

print(f"Observed cells: {master['is_observed'].sum():,} ({100 * master['is_observed'].mean():.1f}%)")

Observed cells: 3,357,774 (69.6%)


### 9. Fill small gaps by carrying the last value forward

In [11]:
# How many hours we allow a value to be carried
CUTOFF = {'heart_rate': 4, 'systolic_bp': 4, 'diastolic_bp': 4, 'mean_bp': 4,
          'spo2': 4, 'respiratory_rate': 4,
          'fio2': 12, 'peep': 12, 'tidal_volume': 12,
          'pao2': 12, 'paco2': 12, 'ph': 12, 'lactate': 12}

grid = master.sort_values(['visit_occurrence_id', 'feature_name', 'hour_bin']).copy()
g = grid.groupby(['visit_occurrence_id', 'feature_name'], sort=False)

# carry the last value forward
grid['value_filled'] = g['value_as_number'].ffill()

# hour of the last real measurement, carried forward the same way
grid['last_obs'] = grid['hour_bin'].where(grid['is_observed'] == 1)
grid['last_obs'] = g['last_obs'].ffill()

# drop fills that reach back too far
grid['gap'] = grid['hour_bin'] - grid['last_obs']
grid.loc[grid['gap'] > grid['feature_name'].map(CUTOFF), 'value_filled'] = np.nan

print(f"Coverage after fill: {100 * grid['value_filled'].notna().mean():.1f}%")

Coverage after fill: 94.7%


### 10. Split patients into train / validation / test

In [12]:
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42
ids = windows['visit_occurrence_id'].unique()

s1 = GroupShuffleSplit(1, train_size=0.70, random_state=RANDOM_STATE)
train_pos, hold_pos = next(s1.split(ids, groups=ids))
train_v, hold_v = ids[train_pos], ids[hold_pos]

s2 = GroupShuffleSplit(1, train_size=0.5, random_state=RANDOM_STATE)
val_pos, test_pos = next(s2.split(hold_v, groups=hold_v))
val_v, test_v = hold_v[val_pos], hold_v[test_pos]

train_set, val_set = set(train_v), set(val_v)
windows['split'] = windows['visit_occurrence_id'].apply(
    lambda v: 'train' if v in train_set else ('val' if v in val_set else 'test'))

print(f'Patients -> train {len(train_v):,} | val {len(val_v):,} | test {len(test_v):,}')

Patients -> train 1,338 | val 287 | test 287


### 11. Fill any leftover gaps with the training median

In [13]:
grid = grid.merge(windows[['visit_occurrence_id', 'split']].drop_duplicates(),
                  on='visit_occurrence_id', how='left')

medians = grid[grid['split'] == 'train'].groupby('feature_name')['value_filled'].median()
grid['value_complete'] = grid['value_filled'].fillna(grid['feature_name'].map(medians))

print(f"Missing left: {grid['value_complete'].isna().sum()}")

Missing left: 0


### 12. Scale each feature (using training data only)

In [14]:
from sklearn.preprocessing import StandardScaler

scalers = {}
grid['value_scaled'] = np.nan
for f in FEATURES:
    train_vals = grid.loc[(grid['feature_name'] == f) & (grid['split'] == 'train'),
                          'value_complete'].values.reshape(-1, 1)
    scalers[f] = StandardScaler().fit(train_vals)

    m = grid['feature_name'] == f
    grid.loc[m, 'value_scaled'] = scalers[f].transform(
        grid.loc[m, 'value_complete'].values.reshape(-1, 1)).ravel()

print('Scaling done.')

Scaling done.


### 13. Turn each window into input and target arrays

AI-assisted: build context/target arrays from an hourly lookup, generated with Claude Opus 4.8 and verified by the author.


In [15]:
lut = grid.set_index(['visit_occurrence_id', 'feature_name', 'hour_bin']).sort_index()
val_lut, obs_lut = lut['value_scaled'], lut['is_observed']

offsets = list(range(-CONTEXT_HOURS, 0))
X_ctx, X_ctx_mask, y_tgt, y_tgt_mask, kept_ids = [], [], [], [], []

for r in windows.itertuples(index=False):
    v, t = r.visit_occurrence_id, r.origin_t

    ctx = np.zeros((CONTEXT_HOURS, len(FEATURES)))
    ctx_m = np.zeros((CONTEXT_HOURS, len(FEATURES)))
    tgt = np.zeros((len(HORIZONS), len(FEATURES)))
    tgt_m = np.zeros((len(HORIZONS), len(FEATURES)))

    for fi, f in enumerate(FEATURES):
        for hi, off in enumerate(offsets):
            ctx[hi, fi] = val_lut[(v, f, t + off)]
            ctx_m[hi, fi] = obs_lut[(v, f, t + off)]
        for hi, hz in enumerate(HORIZONS):
            tgt[hi, fi] = val_lut[(v, f, t + hz)]
            tgt_m[hi, fi] = obs_lut[(v, f, t + hz)]

    X_ctx.append(ctx); X_ctx_mask.append(ctx_m)
    y_tgt.append(tgt); y_tgt_mask.append(tgt_m)
    kept_ids.append(r.window_id)

X_context = np.stack(X_ctx)
X_context_mask = np.stack(X_ctx_mask)
y_target = np.stack(y_tgt)
y_target_mask = np.stack(y_tgt_mask)

print(f'Context: {X_context.shape}  Target: {y_target.shape}')
print(f'Observed targets: {100 * y_target_mask.mean():.1f}%')

Context: (13017, 48, 13)  Target: (13017, 3, 13)
Observed targets: 68.9%


### 14. Save the grid (Tier 1 and Tier 2)

In [16]:
smap = windows.set_index('window_id')['split']
vmap = windows.set_index('window_id')['visit_occurrence_id']
split_per_window = np.array([smap[w] for w in kept_ids])
visit_per_window = np.array([vmap[w] for w in kept_ids])

np.savez_compressed(
    GRID_PATH,
    X_context=X_context, X_context_mask=X_context_mask,
    y_target=y_target, y_target_mask=y_target_mask,
    window_ids=np.array(kept_ids), visit_ids=visit_per_window, split=split_per_window)

print(f'Saved grid -> {GRID_PATH}')

Saved grid -> /content/drive/MyDrive/ventilator_weaning/windows/windows_tier12_grid_imvstart.npz


### 15. Save the triplets (Tier 3)

In [17]:
kept_visits = set(visit_per_window)
max_origin = windows.groupby('visit_occurrence_id')['origin_t'].max()

trip = df[df['visit_occurrence_id'].isin(kept_visits)].copy()
trip = trip[trip['hours_since_imv_start'] >= 0]
trip = trip.merge(max_origin.rename('max_origin'), on='visit_occurrence_id', how='left')
trip = trip[trip['hours_since_imv_start'] <= trip['max_origin']]

triplets = trip[['visit_occurrence_id', 'feature_name', 'hours_since_imv_start', 'value_as_number']]
triplets = triplets.merge(windows[['visit_occurrence_id', 'split']].drop_duplicates(),
                          on='visit_occurrence_id', how='left')
triplets.to_parquet(TRIPLETS_PATH)

print(f'Saved triplets -> {TRIPLETS_PATH} ({len(triplets):,} rows)')

Saved triplets -> /content/drive/MyDrive/ventilator_weaning/windows/windows_tier3_triplets_imvstart.parquet (4,124,550 rows)


### 16. Save the metadata

In [18]:
all_origins = sorted({t for o in origins_by_visit.values() for t in o})
scaler_stats = {f: {'mean': float(scalers[f].mean_[0]),
                    'std': float(scalers[f].scale_[0]),
                    'median_fill': float(medians[f])} for f in FEATURES}

metadata = {
    'anchor': 'imv_start',
    'context_hours': CONTEXT_HOURS,
    'warmup_hours': WARMUP_HOURS,
    'stride_hours': STRIDE_HOURS,
    'max_windows_per_visit': MAX_WINDOWS_PER_VISIT,
    'horizons': HORIZONS,
    'valid_origins': all_origins,
    'features': FEATURES,
    'n_windows': len(kept_ids),
    'n_visits': int(len(kept_visits)),
    'split_counts': {'train_visits': int(len(train_v)),
                     'val_visits': int(len(val_v)),
                     'test_visits': int(len(test_v))},
    'units_note': 'paco2 and pao2 are in kPa (AmsterdamUMCdb convention)',
    'scaler_stats': scaler_stats,
    'random_state': RANDOM_STATE,
}
with open(METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved metadata -> {METADATA_PATH}')

Saved metadata -> /content/drive/MyDrive/ventilator_weaning/windows/windows_metadata_imvstart.json


### 17. Confirm the files saved

In [19]:
check = np.load(GRID_PATH, allow_pickle=True)
assert check['X_context'].shape == X_context.shape
print(f'Done: {X_context.shape[0]:,} windows from {len(kept_visits):,} patients')

Done: 13,017 windows from 1,912 patients


### Summary

Windows are placed by counting forward from ventilation start, so the model only ever sees data it would actually have at that point in time. Patients who were never extubated are included again. T

For Tier 3 (GraFITi): the time column is now `hours_since_imv_start`, and each patient has different window times, so the origin check needs to accept any value from `metadata['valid_origins']`. Tier 1 and Tier 2 only need the file paths updated.